# Surrogate Factory — UCFatigue
## Chapter 7. Model Training
Objectives:
- Train one MLPRegressor per fatigue output.
- Save trained models to artifacts folder.

### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

### 7. Model Training

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
workflow.import_metadata(stage_name="SF_7_Model_Training")

In [ ]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Val_set   = workflow.load_data(workflow.config['job_name'] + '_Val_set.csv')

In [ ]:
from model_training.learn import train
models = train(workflow, Train_set, Val_set)

#### Training Curves
Loss curve per epoch (MLP) and train score per boosting iteration (GradientBoosting).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import joblib
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

models_info = workflow.metadata.get_step_data(['metadata', 'Model_Training', 'Models'])

tab_contents = []
tab_titles   = []

for info in models_info:
    label        = info['label']
    outputs_list = info['outputs']
    model        = joblib.load(info['file'])
    out          = widgets.Output()

    with out:
        if hasattr(model, 'loss_curve_'):   # MLP
            has_val = getattr(model, 'validation_scores_', None) is not None
            fig, axes = plt.subplots(1, 2 if has_val else 1,
                                     figsize=(13 if has_val else 7, 4))
            ax0 = axes[0] if has_val else axes
            ax0.plot(model.loss_curve_, color='steelblue', linewidth=1.2, label='Train loss')
            if model.best_loss_ is not None:
                ax0.axhline(model.best_loss_, color='tomato', linestyle='--', linewidth=0.9,
                            label=f'Best: {model.best_loss_:.5f}')
            ax0.set_xlabel('Epoch')
            ax0.set_ylabel('MSE Loss')
            ax0.set_title(f'Training Loss  (stopped epoch {model.n_iter_} / {model.max_iter})')
            ax0.legend(fontsize=8)
            if has_val:
                axes[1].plot(model.validation_scores_, color='orange', linewidth=1.2,
                             label='Val R²')
                axes[1].set_xlabel('Epoch')
                axes[1].set_ylabel('R² score')
                axes[1].set_title('Validation Score')
                axes[1].legend(fontsize=8)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

        elif hasattr(model, 'estimators_'):  # MultiOutputRegressor (GB)
            ncols  = 4
            nrows  = -(-len(outputs_list) // ncols)
            n_est  = model.estimators_[0].n_estimators
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
            axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
            for i, (est, col) in enumerate(zip(model.estimators_, outputs_list)):
                axes[i].plot(est.train_score_, color='tomato', linewidth=1)
                axes[i].set_title(col, fontsize=9)
                axes[i].set_xlabel('Iteration', fontsize=8)
                axes[i].set_ylabel('MSE', fontsize=8)
            for j in range(len(outputs_list), len(axes)):
                axes[j].set_visible(False)
            fig.suptitle(f'Train Loss per Boosting Iteration  (n_estimators={n_est})', fontsize=11)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    tab_contents.append(out)
    tab_titles.append(label)

tab = widgets.Tab(children=tab_contents)
for i, title in enumerate(tab_titles):
    tab.set_title(i, title)
display(tab)

### Save

In [ ]:
workflow.save_metadata()